In [3]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor
)
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor

In [4]:
df = pd.read_csv('gurgaon_properties_feature_selectionV2.csv')

In [5]:
df

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 11,4.00,5,4,1,Moderately Old,2025.0,0,0,1,Low,Low Floor
1,flat,sector 110,3.30,4,5,3,Relatively New,3032.0,1,0,1,Medium,Mid Floor
2,flat,sector 104,2.30,3,4,3+,Moderately Old,2217.0,1,0,1,Medium,Mid Floor
3,house,sector 50,3.99,4,5,3+,New Property,4500.0,1,0,0,Medium,Mid Floor
4,flat,sector 37,1.45,3,3,3,Relatively New,1711.0,0,0,1,Medium,Mid Floor
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3619,flat,sector 77,1.40,3,3,3,Relatively New,1735.0,0,0,1,Low,Mid Floor
3620,flat,sector 37,1.00,3,3,2,Relatively New,1163.0,0,0,1,Low,Mid Floor
3621,flat,sector 37,0.79,4,4,2,Relatively New,1765.0,0,0,1,Medium,Mid Floor
3622,flat,sector 69,0.60,2,1,1,Relatively New,667.0,0,0,1,Low,High Floor


In [6]:
df['furnishing_type'].value_counts()

furnishing_type
1    2424
0    1012
2     188
Name: count, dtype: int64

In [7]:
# 0 -> unfurnished
# 1 -> semifurnished
# 2 -> furnished
df['furnishing_type'] = df['furnishing_type'].replace({0.0:'unfurnished',1.0:'semifurnished',2.0:'furnished'})

In [8]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 11,4.00,5,4,1,Moderately Old,2025.0,0,0,semifurnished,Low,Low Floor
1,flat,sector 110,3.30,4,5,3,Relatively New,3032.0,1,0,semifurnished,Medium,Mid Floor
2,flat,sector 104,2.30,3,4,3+,Moderately Old,2217.0,1,0,semifurnished,Medium,Mid Floor
3,house,sector 50,3.99,4,5,3+,New Property,4500.0,1,0,unfurnished,Medium,Mid Floor
4,flat,sector 37,1.45,3,3,3,Relatively New,1711.0,0,0,semifurnished,Medium,Mid Floor


In [9]:
X = df.drop(columns=['price'])
y = df['price']

In [10]:
# Applying the log1p transformation to the target variable
y_transformed = np.log1p(y)

### Ordinal Encoding

In [11]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

In [12]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode)
    ], 
    remainder='passthrough'
)

In [13]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [14]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [15]:
scores.mean(),scores.std()

(np.float64(0.7293889892980878), np.float64(0.029922450688131694))

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [17]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [18]:
y_pred = pipeline.predict(X_test)

In [19]:
y_pred = np.expm1(y_pred)

In [20]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.9462556000848452

In [21]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [22]:
model_dict = {
    'linear_reg': LinearRegression(n_jobs=-1),
    'svr': SVR(),
    'ridge': Ridge(),
    'lasso': Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest': RandomForestRegressor(n_jobs=-1),
    'extra trees': ExtraTreesRegressor(n_jobs=-1),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost': XGBRegressor(n_jobs=-1)
}

In [23]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [25]:
model_output

[['linear_reg', np.float64(0.7293889892980878), 0.9462556000848452],
 ['svr', np.float64(0.764431283809355), 0.8294372000263288],
 ['ridge', np.float64(0.7293931817045868), 0.9461612735518887],
 ['lasso', np.float64(0.05031272895892606), 1.4876299629040681],
 ['decision tree', np.float64(0.7789895663126725), 0.6456689065339796],
 ['random forest', np.float64(0.8801110940833723), 0.5175578907968882],
 ['extra trees', np.float64(0.8658380795365994), 0.5582272591775693],
 ['gradient boosting', np.float64(0.8747770533868439), 0.5754403632109576],
 ['adaboost', np.float64(0.7507330905848293), 0.8083413523149221],
 ['mlp', np.float64(0.8030106057360309), 0.7691832058436104],
 ['xgboost', np.float64(0.8889763226984291), 0.5037577896488123]]

In [26]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [27]:
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.888976,0.503758
5,random forest,0.880111,0.517558
6,extra trees,0.865838,0.558227
7,gradient boosting,0.874777,0.575440
4,decision tree,0.778990,0.645669
9,mlp,0.803011,0.769183
8,adaboost,0.750733,0.808341
1,svr,0.764431,0.829437
2,ridge,0.729393,0.946161
0,linear_reg,0.729389,0.946256


### OneHotEncoding

In [29]:
columns_to_encode = ['property_type', 'balcony', 'luxury_category', 'floor_category']

In [30]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first'),['sector','agePossession','furnishing_type'])
    ], 
    remainder='passthrough'
)

In [31]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [33]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [34]:
scores.mean()

np.float64(0.8454575324494302)

In [41]:
scores.std()

np.float64(0.020582068037166876)

In [42]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [43]:
pipeline.fit(X_train,y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [44]:
y_pred = pipeline.predict(X_test)

In [45]:
y_pred = np.expm1(y_pred)

In [46]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.7119871842467536

In [47]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [48]:
model_dict = {
    'linear_reg': LinearRegression(n_jobs=-1),
    'svr': SVR(),
    'ridge': Ridge(),
    'lasso': Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest': RandomForestRegressor(n_jobs=-1),
    'extra trees': ExtraTreesRegressor(n_jobs=-1),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost': XGBRegressor(n_jobs=-1)
}

In [49]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [50]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [51]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.880996,0.504318
10,xgboost,0.886366,0.520204
5,random forest,0.872195,0.527652
1,svr,0.878266,0.560836
9,mlp,0.870015,0.568327
7,gradient boosting,0.857407,0.606522
0,linear_reg,0.845458,0.711987
4,decision tree,0.775586,0.717466
2,ridge,0.843651,0.719926
8,adaboost,0.722678,0.893142


### OneHotEncoding With PCA

In [67]:
# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['sector','agePossession','furnishing_type'])
    ], 
    remainder='passthrough'
)

In [68]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('regressor', LinearRegression())
])

In [69]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [70]:
scores.mean()

np.float64(0.7605128103887947)

In [71]:
scores.std()

np.float64(0.02231530531956265)

In [72]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [73]:
model_dict = {
    'linear_reg': LinearRegression(n_jobs=-1),
    'svr': SVR(),
    'ridge': Ridge(),
    'lasso': Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest': RandomForestRegressor(n_jobs=-1),
    'extra trees': ExtraTreesRegressor(n_jobs=-1),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost': XGBRegressor(n_jobs=-1)
}

In [74]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\neural_network\_multi

In [77]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [76]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.836441,0.665294
9,mlp,0.816193,0.678115
1,svr,0.820818,0.678439
5,random forest,0.823716,0.704478
7,gradient boosting,0.819550,0.704487
10,xgboost,0.828433,0.710384
0,linear_reg,0.760513,0.868666
2,ridge,0.760567,0.868679
8,adaboost,0.689765,0.917017
4,decision tree,0.637590,0.967808


### Target Encoder

In [78]:
!pip install category_encoders

Defaulting to user installation because normal site-packages is not writeable


In [80]:
import category_encoders as ce

columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession','furnishing_type']) ,
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [81]:
# Creating a pipeline
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [82]:
# K-fold cross-validation
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [83]:
scores.mean(),scores.std()

(np.float64(0.8216711424522025), np.float64(0.02568653410723178))

In [85]:
def scorer(model_name, model):
    
    output = []
    
    output.append(model_name)
    
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())
    
    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)
    
    pipeline.fit(X_train,y_train)
    
    y_pred = pipeline.predict(X_test)
    
    y_pred = np.expm1(y_pred)
    
    output.append(mean_absolute_error(np.expm1(y_test),y_pred))
    
    return output
    

In [86]:
model_dict = {
    'linear_reg': LinearRegression(n_jobs=-1),
    'svr': SVR(),
    'ridge': Ridge(),
    'lasso': Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest': RandomForestRegressor(n_jobs=-1),
    'extra trees': ExtraTreesRegressor(n_jobs=-1),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost': XGBRegressor(n_jobs=-1)
}

In [87]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [88]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [89]:
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.897895,0.483392
6,extra trees,0.899158,0.491973
10,xgboost,0.894034,0.502802
7,gradient boosting,0.888021,0.578027
4,decision tree,0.808497,0.677381
9,mlp,0.847525,0.691035
8,adaboost,0.803973,0.789507
0,linear_reg,0.821671,0.792473
2,ridge,0.821690,0.792525
1,svr,0.784022,0.809564


### Hyperparameter Tuning

In [90]:
from sklearn.model_selection import GridSearchCV

In [91]:
param_grid = {
    'regressor__n_estimators': [50, 100, 200, 300],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__max_samples':[0.1, 0.25, 0.5, 1.0],
    'regressor__max_features': ['auto', 'sqrt']
}

In [92]:
import category_encoders as ce

columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession','furnishing_type']) ,
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ], 
    remainder='passthrough'
)

In [93]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor())
])

In [94]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

In [95]:
search = GridSearchCV(pipeline, param_grid, cv=kfold, scoring='r2', n_jobs=-1, verbose=4)

In [96]:
search.fit(X, y_transformed)

Fitting 10 folds for each of 128 candidates, totalling 1280 fits


C:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning: 
640 fits failed out of a total of 1280.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
248 fits failed with the following error:
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\ProgramData\anaconda3\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\ProgramData\anaconda3\Lib\site-packages\sklearn\pipeline.py", line 663, in fit
    self._final_estimator

,estimator,Pipeline(step...Regressor())])
,param_grid,"{'regressor__max_depth': [None, 10, ...], 'regressor__max_features': ['auto', 'sqrt'], 'regressor__max_samples': [0.1, 0.25, ...], 'regressor__n_estimators': [50, 100, ...]}"
,scoring,'r2'
,n_jobs,-1
,refit,True
,cv,KFold(n_split... shuffle=True)
,verbose,4
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...), ...]"


In [98]:
final_pipe = search.best_estimator_

In [99]:
search.best_params_

{'regressor__max_depth': 30,
 'regressor__max_features': 'sqrt',
 'regressor__max_samples': 1.0,
 'regressor__n_estimators': 300}

In [100]:
search.best_score_

np.float64(0.8977429950582219)

In [101]:
final_pipe.fit(X,y_transformed)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


### Exporting the model

In [103]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['sector','agePossession'])
    ], 
    remainder='passthrough'
)

In [104]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=500))
])

In [105]:
pipeline.fit(X,y_transformed)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...), ...]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [106]:
import pickle

with open('pipeline.pkl', 'wb') as file:
    pickle.dump(pipeline, file)

In [107]:
with open('df.pkl', 'wb') as file:
    pickle.dump(X, file)

### Trying out the predictions

In [108]:
X.columns

Index(['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category'],
      dtype='object')

In [109]:
X.iloc[0].values

array(['house', 'sector 11', np.int64(5), np.int64(4), '1',
       'Moderately Old', np.float64(2025.0), np.int64(0), np.int64(0),
       'semifurnished', 'Low', 'Low Floor'], dtype=object)

In [111]:
data = [['house', 'sector 102', 4, 3, '3+', 'New Property', 2750, 0, 0, 'unfurnished', 'Low', 'Low Floor']]
columns = ['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category']
# Convert to DataFrame
one_df = pd.DataFrame(data, columns=columns)

one_df

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 102,4,3,3+,New Property,2750,0,0,unfurnished,Low,Low Floor


In [112]:
np.expm1(pipeline.predict(one_df))

array([3.28207586])